<a href="https://colab.research.google.com/github/aypy01/tensorflow/blob/main/bidirectional_lstm_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDB Sentiment Analysis — Keras

## Introduction


This notebook builds a **binary sentiment classifier** using the IMDB movie review dataset.  
Goal: predict if a review is **positive** or **negative**.

### Dataset
- 50,000 reviews: 25k train, 25k test
- Reviews encoded as integer sequences
- Sequences padded/truncated to a fixed length
- 20% of training data used for validation

### Model
- **Embedding layer**: converts integers → 128-dim vectors
- **Two Bidirectional LSTMs**: capture forward + backward context
- **Dense output layer** with sigmoid: predicts sentiment probability

### Training
- Loss: `binary_crossentropy`  
- Optimizer: `adam`  
- Batch size: 32, Epochs: 5  
- Metric: Accuracy (~83% expected)

### Notes
- Accuracy can improve with longer sequences, larger vocabulary, dropout, or pretrained embeddings.
- Test set is only evaluated after training to avoid data leakage.


##Importation

In [10]:
#Importing every library for this
#Only keras and no tensorflow
import numpy as np
import keras
from keras import layers



## Load the IMDB movie review sentiment data

In [11]:
#Features parameters
max_features = 20000  # Only consider the top 20k words
maxlen = 200  # Only consider the first 200 words of each movie review

In [12]:

# keeps only top N words; rest discarded
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features )

# x_train = list of reviews (encoded as sequences of word indices)
# y_train = list of labels (1 = positive, 0 = negative)
# SAME FOR x_val and y_val

print(len(x_train), "Training sequences")
print(len(x_test), "Test sequences")

#Padding is required as the reviews are no in same lenghth:
#Padding if words > 200 the flusg and if the words<200 it will add 0,0,0...
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_test = keras.utils.pad_sequences(x_test, maxlen=maxlen)


25000 Training sequences
25000 Test sequences


## Build the model

In [13]:
# Input layer
# Accepts a variable-length sequence of integer word IDs.
# shape=(None,) : sequence length is flexible.
# dtype=int32 :required because the Embedding layer only accepts integer indices.
inputs = keras.Input(shape=(None,), dtype="int32")

# Embedding layer
# Converts each integer word ID into a 128-dimensional learned vector.
# Output shape becomes: (batch_size, sequence_length, 128)
# This transforms discrete words into continuous float vectors the LSTM can use.
x = layers.Embedding(max_features, 128)(inputs)

# First Bidirectional LSTM
# return_sequences=True returns a sequence (one output per time step)
# Needed when stacking LSTMs so the second LSTM can process the full sequence.
x1 = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)

# Second Bidirectional LSTM
# return_sequences=False (default) returns a single vector summarizing the sequence.
# This becomes the sentence-level representation.
x2 = layers.Bidirectional(layers.LSTM(64))(x1)

# Output layer
# Dense(1) with sigmoid binary classification: positive (1) or negative (0)
outputs = layers.Dense(1, activation="sigmoid")(x2)

model = keras.Model(inputs, outputs)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

##Compile

In [14]:
model.compile(optimizer="adam",
              loss="binary_crossentropy", #coz its between 2 value 0 and 1
              metrics=["accuracy"])


##Taining/Fit

In [19]:
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_split=0.2, verbose=1)

Epoch 1/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.9835 - loss: 0.0515 - val_accuracy: 0.8646 - val_loss: 0.4608
Epoch 2/2
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.9910 - loss: 0.0312 - val_accuracy: 0.8668 - val_loss: 0.5377


##Evaluate

In [20]:
#Evaluate
model.evaluate(x_test,y_test,batch_size=32,verbose=1)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8518 - loss: 0.6026


[0.5931116938591003, 0.8535199761390686]

##Saving

In [21]:
model.save("sentiments_biderectional.keras")